# Train Coffee Roast Classifier สำหรับ Raspberry Pi 5

Notebook นี้ฝึกโมเดลจำแนกระดับการคั่ว **Light / Medium / Dark**, ประเมินบน test split ที่แยกจาก validation และส่งออกเป็น TFLite แบบ full INT8 (uint8 input/output) สำหรับ MQTT inference service บน Raspberry Pi 5

> โฟลเดอร์ `Green` จะไม่ถูกนำมาใช้ เพราะโจทย์นี้จำแนกเฉพาะเมล็ดที่คั่วแล้ว

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import tensorflow as tf

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
from train_rpi_tflite import (
    CLASSES, build_model, compile_model, evaluate_keras, evaluate_tflite,
    export_int8, load_split
)

DATA_ROOT = ROOT / 'datasets' / 'Coffee Bean Roast.v2i.folder'
OUTPUT_DIR = ROOT / 'src'
REPORT_PATH = ROOT / 'reports' / 'rpi_model_metrics.json'
BATCH_SIZE, SEED = 32, 2026
EPOCHS, FINE_TUNE_EPOCHS = 8, 3
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Project:', ROOT)

## 1. โหลดข้อมูล
ใช้ `train` สำหรับปรับน้ำหนัก, `valid` สำหรับ early stopping และใช้ `test` เพียงครั้งเดียวตอนประเมินผลสุดท้าย

In [ ]:
train_ds = load_split(DATA_ROOT, 'train', BATCH_SIZE, SEED, True)
valid_ds = load_split(DATA_ROOT, 'valid', BATCH_SIZE, SEED, False)
test_ds = load_split(DATA_ROOT, 'test', BATCH_SIZE, SEED, False)

images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for image, label, ax in zip(images[:8], labels[:8], axes.flat):
    ax.imshow(tf.cast(image, tf.uint8))
    ax.set_title(CLASSES[int(label)])
    ax.axis('off')
plt.tight_layout()

## 2. สร้างโมเดล MobileNetV2
ถ้ามีโมเดลเดิมที่ compatible จะใช้เป็นจุดเริ่มต้นเพื่อลดเวลา train; ถ้าไม่มีจะโหลดน้ำหนัก ImageNet

In [ ]:
initial_path = ROOT / 'src' / 'coffee_roast_model.keras'
model, base_model = build_model(initial_path if initial_path.exists() else None)
compile_model(model, 3e-4)
model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
head_history = model.fit(
    train_ds, validation_data=valid_ds, epochs=EPOCHS,
    callbacks=[early_stop], verbose=2
)

## 3. Fine-tune ชั้นท้าย
ปลดล็อก 30 ชั้นสุดท้ายของ MobileNetV2 และใช้ learning rate ต่ำเพื่อปรับให้เข้ากับภาพกาแฟโดยไม่ทำลายน้ำหนักเดิม

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
compile_model(model, 1e-5)
fine_history = model.fit(
    train_ds, validation_data=valid_ds, epochs=FINE_TUNE_EPOCHS,
    callbacks=[early_stop], verbose=2
)

## 4. Export เป็น `.keras` และ TFLite INT8
Representative samples ทำให้ weights และ operations ภายในเป็น INT8 ส่วน input/output เป็น uint8 จึงเหมาะกับภาพจากกล้องและ runtime ขนาดเล็กบน Pi

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
keras_path = OUTPUT_DIR / 'coffee_roast_rpi.keras'
tflite_path = OUTPUT_DIR / 'coffee_roast_rpi_int8.tflite'
labels_path = OUTPUT_DIR / 'coffee_roast_labels.json'
model.save(keras_path)
export_int8(model, train_ds.unbatch().batch(1), tflite_path)
labels_path.write_text(json.dumps(list(CLASSES), indent=2), encoding='utf-8')
print(f'{tflite_path.name}: {tflite_path.stat().st_size / 1024 / 1024:.2f} MiB')

## 5. ตรวจว่าไฟล์ TFLite ใช้งานได้จริง
รายงานนี้เปรียบเทียบผลของ Keras และ TFLite บน test split พร้อมบันทึก input dtype/quantization metadata

In [ ]:
keras_result = evaluate_keras(model, test_ds)
tflite_result = evaluate_tflite(tflite_path, test_ds)
report = {
    'classes': list(CLASSES), 'keras_test': keras_result, 'tflite_test': tflite_result,
    'artifacts': {'keras': str(keras_path), 'tflite': str(tflite_path), 'labels': str(labels_path)},
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Keras accuracy :', keras_result['classification_report']['accuracy'])
print('TFLite accuracy:', tflite_result['classification_report']['accuracy'])
print('TFLite input   :', tflite_result['input'])
print('TFLite output  :', tflite_result['output'])

## 6. Export production ensemble สำหรับภาพกองเมล็ด
โมเดลที่เพิ่ง train เหมาะกับโดเมนของ dataset ใหม่ ส่วน deployment เริ่มต้นใช้ ensemble ของโมเดลภาพกองเมล็ดสองตัวในโปรเจกต์ รวมเป็นไฟล์ TFLite INT8 ไฟล์เดียว เพื่อรองรับภาพจริงได้ครอบคลุมกว่า

In [ ]:
import subprocess
subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'export_rpi_ensemble.py')],
    cwd=ROOT, check=True
)
ensemble_path = OUTPUT_DIR / 'coffee_roast_rpi_ensemble_int8.tflite'
print(f'{ensemble_path.name}: {ensemble_path.stat().st_size / 1024 / 1024:.2f} MiB')

## 7. นำไปรันกับ MQTT Broker บน Raspberry Pi 5

คัดลอกไฟล์โมเดล, labels, requirements และ `mqtt_tflite_service.py` ไปยัง Pi แล้วรัน:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r src/requirements-rpi.txt
python scripts/mqtt_tflite_service.py --host localhost
```

service ใช้ `coffee_roast_rpi_ensemble_int8.tflite` เป็นค่าเริ่มต้น, subscribe ที่ `coffee/image` (raw JPEG/PNG หรือ JSON base64) และ publish JSON ที่ `coffee/prediction`

In [ ]:
# ตัวอย่างส่งภาพจากเครื่องอื่น
# import paho.mqtt.publish as publish
# image_bytes = (ROOT / 'test_samples' / 'dark-roast.png').read_bytes()
# publish.single('coffee/image', image_bytes, hostname='RASPBERRY_PI_IP', qos=1)